__Paper__: Using neural networks as an alternative to air dispersion modeling in environmental impact assessment.

__Authors__: Mateo Concha and Gonzalo A. Ruz.

__Description__: Code used for the paper development.

In [2]:
pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# --- Load datasets (adjust folder if needed) ---
train_data     = pd.read_csv("train.csv")
validate_data  = pd.read_csv("validation.csv")
test_FFEE_data = pd.read_csv("test_FFEE.csv")
test_CV_data   = pd.read_csv("test_CV.csv")

# --- Explicit target columns
target_ffee = "FFEE"
target_cv   = "Chacaya"

# --- Sanity checks (gives a clear error message if files differ) ---
for name, df, target in [
    ("train", train_data, target_ffee),
    ("validation", validate_data, target_ffee),
    ("test_FFEE", test_FFEE_data, target_ffee),
    ("test_CV", test_CV_data, target_cv),
]:
    if target not in df.columns:
        raise KeyError(
            f"{name}: expected target column '{target}' not found.\n"
            f"Available columns: {list(df.columns)}"
        )

# --- Split into X/y ---
X_train, y_train = train_data.drop(columns=[target_ffee]), train_data[target_ffee]
X_val,   y_val   = validate_data.drop(columns=[target_ffee]), validate_data[target_ffee]

X_test_FFEE, y_test_FFEE = test_FFEE_data.drop(columns=[target_ffee]), test_FFEE_data[target_ffee]
X_test_CV,   y_test_CV   = test_CV_data.drop(columns=[target_cv]),   test_CV_data[target_cv]

print("Shapes:",
      "train", X_train.shape,
      "val", X_val.shape,
      "test_FFEE", X_test_FFEE.shape,
      "test_CV", X_test_CV.shape)

# --- Scaling (fit on TRAIN only; apply to all) ---
scaler = MinMaxScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_val   = scaler.transform(X_val)
X_test_FFEE = scaler.transform(X_test_FFEE)
X_test_CV   = scaler.transform(X_test_CV)


Shapes: train (6000, 24) val (1000, 24) test_FFEE (1750, 24) test_CV (2947, 24)


In [7]:
pipe = Pipeline([
    ("scaler", StandardScaler()),  
    ("rf", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

param_distributions = {
    "rf__n_estimators": randint(50, 201),
    "rf__max_depth": randint(5, 41),         
    "rf__max_features": randint(3, X_train.shape[1] + 1),
    "rf__min_samples_split": randint(2, 11),
    "rf__min_samples_leaf": randint(1, 6),
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=60,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
print("Best CV params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

# Evaluate chosen model on the fixed validation.csv
best_model = search.best_estimator_
val_pred = best_model.predict(X_val)
report_metrics("Validation (fixed split)", y_val, val_pred)


Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best CV params: {'rf__max_depth': 7, 'rf__max_features': 5, 'rf__min_samples_leaf': 2, 'rf__min_samples_split': 5, 'rf__n_estimators': 156}
Best CV RMSE: 14.490260009224752
Validation (fixed split): RMSE=7.910 | MAE=5.897


In [9]:
# Refit on train+val (for final baseline numbers)
X_tr_all = np.vstack([X_train, X_val])
y_tr_all = np.concatenate([y_train, y_val])

final_model = search.best_estimator_
final_model.fit(X_tr_all, y_tr_all)

pred_test_f = final_model.predict(X_test_FFEE)
pred_test_cv = final_model.predict(X_test_CV)

report_metrics("Test FFEE", y_test_FFEE, pred_test_f)
report_metrics("Test CV", y_test_CV, pred_test_cv)


Test FFEE: RMSE=4.606 | MAE=2.964
Test CV: RMSE=8.694 | MAE=7.955
